# ROGII SEL15 forced selector rerun

Self-owned rerun of the public SEL15/gold96 blend for submission validation.

# ROGII Aiden Gold96 Blend 8.107 Repeat

Submission notebook based on `references/rogii_tree_models_and_physics_blend-8.107`.

Goal: reproduce the reference logic and keep the run easy to audit. The predictive path is:

- discover and align an external `gold96` notebook output submission;
- generate the Aiden SEL15 / physical-model submission dynamically from competition data;
- write final `submission.csv` as `0.60 * Aiden_SEL15 + 0.40 * gold96`;
- write manifests and visual diagnostics after the final submission.


## Run Contract

This notebook is self-contained except for the external `gold96` output dataset. On Kaggle, attach the competition dataset and the output dataset whose path contains one of these needle pairs:

- `beicicc` + `nannicha`;
- `nannicha` + `dwt`;
- `rogii-0525` + `nannicha`.

If that output dataset is not attached, the notebook should fail early before producing an accidental fallback submission.

If your attached output dataset has a different slug, the notebook will also accept it when it is the only `submission.csv` found under `/kaggle/input`. If several candidates exist, edit `GOLD96_PATH_HINTS` in the first code cell so exactly one candidate matches.


## Solution Approach Diagram

The diagram below shows the exact submission path for this notebook. It is audit-only and does not change predictions.


In [ ]:
from IPython.display import HTML, display

display(HTML(r"""
<div style="max-width:1180px;margin:10px 0 18px 0;">
  <svg viewBox="0 0 1180 620" width="100%" role="img" aria-label="ROGII solution approach diagram" xmlns="http://www.w3.org/2000/svg">
    <defs>
      <marker id="arrow" markerWidth="12" markerHeight="12" refX="10" refY="6" orient="auto" markerUnits="strokeWidth">
        <path d="M2,2 L10,6 L2,10 Z" fill="#334155" />
      </marker>
      <filter id="shadow" x="-20%" y="-20%" width="140%" height="140%">
        <feDropShadow dx="0" dy="2" stdDeviation="2" flood-color="#0f172a" flood-opacity="0.16"/>
      </filter>
      <style>
        .title { font: 700 26px -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; fill:#111827; }
        .subtitle { font: 500 14px -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; fill:#475569; }
        .box-title { font: 700 16px -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; fill:#0f172a; }
        .box-text { font: 500 13px -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; fill:#334155; }
        .small { font: 600 12px -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; fill:#475569; }
        .guard { font: 700 12px -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; fill:#7c2d12; }
        .arrow { stroke:#334155; stroke-width:2.2; fill:none; marker-end:url(#arrow); }
        .soft-arrow { stroke:#64748b; stroke-width:2; fill:none; marker-end:url(#arrow); stroke-dasharray:7 6; }
      </style>
    </defs>

    <rect x="0" y="0" width="1180" height="620" rx="14" fill="#f8fafc"/>
    <text x="40" y="48" class="title">ROGII 8.107 repeat: submission flow</text>
    <text x="40" y="74" class="subtitle">Preserve the reference prediction logic, verify inputs early, blend two aligned submission streams, then audit component disagreement.</text>

    <rect x="40" y="112" width="260" height="112" rx="10" fill="#ffffff" stroke="#cbd5e1" filter="url(#shadow)"/>
    <text x="62" y="145" class="box-title">Competition data</text>
    <text x="62" y="172" class="box-text">sample_submission.csv</text>
    <text x="62" y="193" class="box-text">train/test horizontal wells</text>
    <text x="62" y="214" class="box-text">typewell GR reference</text>

    <rect x="40" y="292" width="260" height="112" rx="10" fill="#fff7ed" stroke="#fed7aa" filter="url(#shadow)"/>
    <text x="62" y="325" class="box-title">External gold96 output</text>
    <text x="62" y="352" class="box-text">/kaggle/input/**/submission.csv</text>
    <text x="62" y="373" class="box-text">matched by nannicha needles</text>
    <text x="62" y="394" class="guard">Fail early if missing</text>

    <rect x="390" y="96" width="300" height="144" rx="10" fill="#ecfeff" stroke="#67e8f9" filter="url(#shadow)"/>
    <text x="416" y="132" class="box-title">Aiden SEL15 / physical branch</text>
    <text x="416" y="160" class="box-text">If train-overlap exists: physical TVT</text>
    <text x="416" y="181" class="box-text">Else: PF scales 3, 5, 8, 12</text>
    <text x="416" y="202" class="box-text">Beam ensemble plus last-known hold</text>
    <text x="416" y="223" class="box-text">Writes temporary submission.csv</text>

    <rect x="390" y="292" width="300" height="112" rx="10" fill="#fefce8" stroke="#fde68a" filter="url(#shadow)"/>
    <text x="416" y="326" class="box-title">Gold96 artifact branch</text>
    <text x="416" y="354" class="box-text">Infer prediction column</text>
    <text x="416" y="375" class="box-text">Align by sample ids</text>
    <text x="416" y="396" class="box-text">Writes gold96_submission.csv</text>

    <rect x="782" y="178" width="270" height="122" rx="10" fill="#f0fdf4" stroke="#86efac" filter="url(#shadow)"/>
    <text x="808" y="213" class="box-title">Alignment and blend</text>
    <text x="808" y="241" class="box-text">Validate one-to-one ids</text>
    <text x="808" y="262" class="box-text">0.60 * Aiden_SEL15</text>
    <text x="808" y="283" class="box-text">0.40 * gold96</text>

    <rect x="782" y="382" width="270" height="112" rx="10" fill="#eff6ff" stroke="#93c5fd" filter="url(#shadow)"/>
    <text x="808" y="416" class="box-title">Audit diagnostics</text>
    <text x="808" y="444" class="box-text">blend_manifest.json</text>
    <text x="808" y="465" class="box-text">per-well diff table and plots</text>
    <text x="808" y="486" class="box-text">run diagnostics JSON</text>

    <rect x="908" y="535" width="174" height="48" rx="10" fill="#0f172a" filter="url(#shadow)"/>
    <text x="936" y="565" style="font:700 16px -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; fill:#ffffff;">submission.csv</text>

    <path class="arrow" d="M300 168 C336 168, 352 168, 390 168"/>
    <path class="arrow" d="M300 348 C336 348, 352 348, 390 348"/>
    <path class="arrow" d="M690 168 C728 168, 744 202, 782 224"/>
    <path class="arrow" d="M690 348 C728 348, 744 278, 782 258"/>
    <path class="arrow" d="M917 300 L917 382"/>
    <path class="arrow" d="M917 494 C928 516, 952 526, 975 535"/>
    <path class="soft-arrow" d="M917 300 C950 340, 980 425, 1000 535"/>

    <rect x="40" y="468" width="640" height="74" rx="10" fill="#ffffff" stroke="#cbd5e1"/>
    <text x="62" y="499" class="box-title">Experiment principle</text>
    <text x="62" y="525" class="box-text">First reproduce the strongest public reference exactly. Only after the score is known, change one controlled lever at a time.</text>
  </svg>
</div>
"""))


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")


def _find_sample_submission() -> Path:
    hits = sorted(ROOT.rglob("sample_submission.csv"))
    preferred = [p for p in hits if "rogii-wellbore-geology-prediction" in str(p)]
    if preferred:
        return preferred[0]
    if hits:
        return hits[0]
    raise FileNotFoundError("sample_submission.csv not found under /kaggle/input")


GOLD96_PATH_HINTS = [
    ("beicicc", "nannicha"),
    ("nannicha", "dwt"),
    ("rogii-0525", "nannicha"),
]
ALLOW_SINGLE_GOLD96_CANDIDATE = True


def _find_gold96_submission() -> Path:
    submissions = sorted(ROOT.rglob("submission.csv"))
    print(f"Gold96 submission candidates found under {ROOT}: {len(submissions)}")
    for needles in GOLD96_PATH_HINTS:
        matches = [p for p in submissions if all(n in str(p).lower() for n in needles)]
        if len(matches) == 1:
            print(f"Selected gold96 by hints {needles}: {matches[0]}")
            return matches[0]
        if len(matches) > 1:
            raise FileNotFoundError(
                "Multiple gold96 candidates matched hints "
                f"{needles}: {[str(p) for p in matches]}"
            )
    if ALLOW_SINGLE_GOLD96_CANDIDATE and len(submissions) == 1:
        print(f"Selected the only available gold96 candidate: {submissions[0]}")
        return submissions[0]
    all_candidates = [str(p) for p in submissions]
    raise FileNotFoundError(
        "Could not select the gold96 notebook output submission.csv under /kaggle/input. "
        "Attach the external output dataset used by the 8.107 reference, or edit "
        "GOLD96_PATH_HINTS in this cell so exactly one candidate matches. "
        f"Candidates: {all_candidates}"
    )


def _prediction_column(frame: pd.DataFrame) -> str:
    lower = {str(col).lower(): col for col in frame.columns}
    for name in ("tvt", "pred_tvt", "prediction"):
        if name in lower:
            return str(lower[name])
    numeric = [col for col in frame.columns if col != "id" and pd.api.types.is_numeric_dtype(frame[col])]
    if len(numeric) == 1:
        return str(numeric[0])
    raise ValueError(f"Cannot infer prediction column from columns={frame.columns.tolist()}")


def _write_gold96_artifact() -> None:
    sample_path = _find_sample_submission()
    gold_path = _find_gold96_submission()
    sample = pd.read_csv(sample_path)
    gold = pd.read_csv(gold_path)
    if "id" not in sample.columns or "id" not in gold.columns:
        raise KeyError("sample/gold submissions must both contain an id column")
    pred_col = _prediction_column(gold)
    compact = gold[["id", pred_col]].rename(columns={pred_col: "tvt"})
    if compact["id"].duplicated().any():
        dupes = compact.loc[compact["id"].duplicated(), "id"].head(10).tolist()
        raise ValueError(f"gold96 submission has duplicated ids: {dupes}")
    aligned = sample[["id"]].merge(compact, on="id", how="left", validate="one_to_one")
    if aligned["tvt"].isna().any():
        missing = aligned.loc[aligned["tvt"].isna(), "id"].head(10).tolist()
        raise ValueError(f"gold96 submission missing sample ids: {missing}")
    aligned["tvt"] = pd.to_numeric(aligned["tvt"], errors="raise").astype(float)
    aligned[["id", "tvt"]].to_csv(OUT / "gold96_submission.csv", index=False)
    # The downstream blend code reads a relative path.
    if (OUT / "gold96_submission.csv").resolve() != Path("gold96_submission.csv").resolve():
        aligned[["id", "tvt"]].to_csv("gold96_submission.csv", index=False)
    manifest = {
        "schema_version": "gold96_kernel_output_artifact_v1",
        "sample_submission": str(sample_path),
        "gold96_submission": str(gold_path),
        "rows": int(len(aligned)),
    }
    (OUT / "gold96_artifact_manifest.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8"
    )
    print(json.dumps(manifest, indent=2, sort_keys=True))


_write_gold96_artifact()


## Dynamic Aiden SEL15 / Physical Component

This section is copied from the `8.107` reference. It computes the Aiden component and writes a temporary `submission.csv`; the final blend cell overwrites that file with the blended submission.


In [ ]:
# --- Aiden SEL15 selector physical model, run dynamically on the same Kaggle runtime data. ---
"""
ROGII Wellbore Geology Prediction â€” v12
DSC 204A Final Project

Strategy:
  - Visible training wells: physical model (RMSE ~0.007 ft)
  - Hidden test wells: PF ensemble ONLY â€” 128 seeds, lik-weighted (scale=5)
    init_spread=2.0 ft (wider initial particle spread)
    GR interpolated before PF (fills NaN gaps so PF always has observations)
    Local avg: 4.71 ft (vs 5.95 ft without GR interpolation)
    Key insight: 000d7d20 has 47% NaN GR in prediction â€” interpolation critical
"""

import os, glob, warnings
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

warnings.filterwarnings('ignore')

SELECTOR_N_EVAL_THRESHOLD = 4840.0
SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)
SELECTOR_BIN_VARIANTS = {
    0: "pf_scale_5_hold_0.2",
    1: "pf_scale_3_hold_0.15",
    2: "pf_scale_12_beam_0.2_hold_0.15",
    3: "pf_scale_5_hold_0.15",
    4: "pf_scale_5_beam_0.05_hold_0.05",
    5: "pf_scale_12_beam_0.2_hold_0.05",
}
SELECTOR_GLOBAL_VARIANT = "pf_scale_8_hold_0.2"
SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)


def find_input_dir():
    for c in ['/kaggle/input/rogii-wellbore-geology-prediction',
              '/kaggle/input/competitions/rogii-wellbore-geology-prediction']:
        if os.path.isdir(c):
            print(f'INPUT_DIR={c}')
            return c
    hits = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
    if hits:
        d = os.path.dirname(hits[0])
        print(f'Discovered INPUT_DIR={d}')
        return d
    raise FileNotFoundError('Cannot locate competition data')


INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, 'train')
TEST_DIR  = os.path.join(INPUT_DIR, 'test')

_hw_files  = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
TEST_WELLS = [os.path.basename(f).split('__')[0] for f in _hw_files]
print(f'Test wells: {TEST_WELLS}')


def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw


def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
    tw_g = tw_tr.dropna(subset=['Geology'])
    ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    if np.isnan(ref_tvt):
        ref_col = tw_g['Geology'].iloc[0]
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
    return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset


def run_particle_filter(hw, tw, n_particles=500, seed=42):
    """Conservative PF. Returns (predictions_array, total_log_likelihood)."""
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input'])
    last_Z   = float(last['Z'])
    last_MD  = float(last['MD'])

    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    N   = n_particles
    rng = np.random.default_rng(seed)
    ls   = last_tvt + last_Z
    pos  = ls + 3.0 * rng.standard_normal(N)  # aiwody/needless spread3 reduces hidden-branch overfitting noise
    rate = ir + 0.01 * rng.standard_normal(N)
    w    = np.ones(N) / N

    MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5

    md_v = ev['MD'].values.astype(float)
    z_v  = ev['Z'].values.astype(float)
    # Interpolate GR gaps before tracking â€” critical for wells with high NaN fraction
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    gr_v = gr_interp.values.astype(float)[ev.index]

    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev))
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos   = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.exp(-0.5 * np.minimum(d**2, 600.))
        lk = np.maximum(lk, 1e-300)
        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        n_eff = 1.0 / (w**2).sum()
        if n_eff < RESAMP * N:
            cum = np.cumsum(w)
            u0  = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos  = pos[idx]  + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w    = np.ones(N) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    """
    128-seed lik-weighted PF ensemble.
    More seeds â†’ better coverage of the TVT exploration space.
    """
    preds = []
    liks  = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)

    liks   = np.array(liks)
    liks_n = liks - liks.max()
    weights = np.exp(liks_n / scale)
    weights /= weights.sum()

    return (weights[:, None] * np.stack(preds, 0)).sum(0)


def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
    preds = []
    liks = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)
    pred_arr = np.stack(preds, 0)
    liks = np.array(liks)
    liks_n = liks - liks.max()
    out = {}
    for scale in scales:
        weights = np.exp(liks_n / float(scale))
        weights /= weights.sum()
        out[f"pf_scale_{scale:g}"] = (weights[:, None] * pred_arr).sum(0)
    out["pf_mean"] = pred_arr.mean(0)
    return out


# 14 beam configs: original 7 + 7 new ones exploring broader parameter space
BEAM_CONFIGS = [
    # Original 7 configs (from ajayrao43)
    (10, 20.0, 144.0, 2),
    (10,  8.0,  64.0, 2),
    ( 8, 35.0, 220.0, 1),
    (10, 14.0,  90.0, 5),
    (20,  4.0,  36.0, 3),
    (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
    # 7 new configs: wider beam, different motion/error scales
    (20, 30.0, 200.0, 2),
    (15, 10.0,  80.0, 4),
    (25,  6.0,  50.0, 3),
    (10, 40.0, 300.0, 1),
    (12, 18.0, 120.0, 5),
    (30,  8.0,  70.0, 2),
    (10, 50.0, 400.0, 0),
]


def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    """Vectorized beam search for TVT tracking via GR matching."""
    n  = len(hgr)
    nt = len(tw_tvt)
    if n == 0:
        return np.array([last_tvt])

    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()

    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    MC    = mc * np.array([2., 1., 0., 1., 2.])

    bidx  = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf)
    bcost[0] = 0.
    bn = 1

    result = np.zeros(n)

    for step in range(n):
        gv = sgr[step]
        ni = bidx[:bn, None] + MOVES[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)

        gr_e = (gv - tw_gr[ci])**2 / es
        tot  = bcost[:bn, None] + gr_e + MC[None, :]
        tot  = np.where(valid, tot, np.inf)

        ni_f  = ni.flatten()
        tot_f = tot.flatten()
        vf    = valid.flatten()
        ni_f  = ni_f[vf]
        tot_f = tot_f[vf]

        order = np.argsort(tot_f)
        ni_s  = ni_f[order]
        tot_s = tot_f[order]

        _, first = np.unique(ni_s, return_index=True)
        ni_u  = ni_s[first]
        tot_u = tot_s[first]

        kept = min(bs, len(ni_u))
        top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
        top  = top[np.argsort(tot_u[top])]

        bidx[:kept]  = ni_u[top]
        bcost[:kept] = tot_u[top]
        if kept < bs:
            bidx[kept:]  = bidx[kept - 1]
            bcost[kept:] = np.inf
        bn = kept

        result[step] = tw_tvt[bidx[0]]

    return result


def run_beam_ensemble(hw, tw):
    """Average 14 beam-search configs."""
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    last_tvt = float(kn.iloc[-1]['TVT_input'])
    tw_s  = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr    = gr_all[ev.index]

    beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
                    for (bs, mc, es, r) in BEAM_CONFIGS]

    beam_mean = np.stack(beam_results, 0).mean(0)

    out = hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)] = beam_mean
    return out


def selector_well_code(hw):
    eval_mask = hw['TVT_input'].isna().to_numpy()
    n_eval = float(eval_mask.sum())
    z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
    z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
    n_bin = int(n_eval > SELECTOR_N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
    code = n_bin + 2 * z_bin
    variant = SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
    return code, variant, n_eval, z_span


def parse_selector_variant(name):
    parts = name.split('_')
    scale = float(parts[2])
    beam_weight = 0.0
    hold_weight = 0.0
    if 'beam' in parts:
        beam_weight = float(parts[parts.index('beam') + 1])
    if 'hold' in parts:
        hold_weight = float(parts[parts.index('hold') + 1])
    return scale, beam_weight, hold_weight


def apply_selector_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
    scale, beam_weight, hold_weight = parse_selector_variant(name)
    base = pf_by_scale.get(f"pf_scale_{scale:g}")
    if base is None:
        base = pf_by_scale[SELECTOR_GLOBAL_VARIANT.split('_beam_')[0].split('_hold_')[0]]
    pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
    pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
    return pred


sample = pd.read_csv(os.path.join(INPUT_DIR, 'sample_submission.csv'))
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str[9:].astype(int)

train_wids = set(
    os.path.basename(f).split('__')[0]
    for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))
)
print(f'Training wells available: {len(train_wids)}')

rows = []
for wid in TEST_WELLS:
    print(f'\nProcessing {wid}...')
    hw_te, tw_te = load_well(wid, 'test')

    tvt_phys = None
    hw_tr    = None
    tw_tr    = None

    # Physical model for visible wells
    if wid in train_wids:
        try:
            hw_tr, tw_tr = load_well(wid, 'train')
            hw_te['TVT_input'] = hw_tr['TVT_input'].values
            tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
            print(f'  Physical model OK')
        except Exception as e:
            print(f'  Physical model failed: {e}')
            tvt_phys = None

    selector_code, selector_variant, selector_n_eval, selector_z_span = selector_well_code(hw_te)

    # 128-seed likelihood-weighted PF ensemble, cached across selector scales.
    try:
        tw_ref = tw_tr if tw_tr is not None else tw_te
        pf_by_scale = run_pf_lik_ensemble_scales(hw_te, tw_ref, n_particles=500, n_seeds=64)
        tvt_pf = pf_by_scale["pf_scale_8"]
        print(f'  PF 64-seed spread3 lik-ensemble OK scales={SELECTOR_SCALES}')
    except Exception as e:
        print(f'  PF failed: {e}')
        last_known = hw_te['TVT_input'].dropna()
        last_val   = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
        tvt_pf = hw_te['TVT_input'].fillna(last_val).values.astype(float)
        pf_by_scale = {f"pf_scale_{scale:g}": tvt_pf.copy() for scale in SELECTOR_SCALES}

    try:
        tw_ref = tw_tr if tw_tr is not None else tw_te
        tvt_beam = run_beam_ensemble(hw_te, tw_ref)
        print(f'  Beam 14-config ensemble OK')
    except Exception as e:
        print(f'  Beam failed: {e}')
        tvt_beam = tvt_pf.copy()

    last_known = hw_te['TVT_input'].dropna()
    last_known_tvt = float(last_known.iloc[-1]) if len(last_known) > 0 else float(np.nanmean(tvt_pf))
    tvt_selector = apply_selector_variant(selector_variant, pf_by_scale, tvt_beam, last_known_tvt)
    print(
        f'  Selector code={selector_code} variant={selector_variant} '
        f'n_eval={selector_n_eval:.0f} z_span={selector_z_span:.3f}'
    )

    ws = sample[sample['well'] == wid]
    for _, row in ws.iterrows():
        ridx = int(row['row_idx'])
        if tvt_phys is not None:
            # Visible well: physical model is primary (RMSE ~0.007 ft)
            tvt_val = float(tvt_phys.iloc[ridx])
        else:
            # Hidden well: targetless selector from whole-well CV.
            tvt_val = float(tvt_selector[ridx])
        rows.append({'id': row['id'], 'tvt': tvt_val})
    print(f'  Added {len(ws)} rows')

submission = pd.DataFrame(rows)
submission.to_csv('submission.csv', index=False)
print(f'\nDone: {len(submission)} rows')
print(submission.head())


## Final Blend And Diagnostics

The final Kaggle file is `submission.csv`. The plots below are audit-only and do not alter predictions.


In [ ]:
# --- Conservative blend: Aiden SEL15 selector primary, clean gold96 low-dose support. ---
import json as _blend_json
import numpy as _blend_np
import pandas as _blend_pd
from pathlib import Path as _BlendPath

AIDEN_SEL15_WEIGHT = 0.60
GOLD96_WEIGHT = 1.0 - AIDEN_SEL15_WEIGHT

aiden_sel15_sub = _blend_pd.read_csv("submission.csv")
gold96_sub = _blend_pd.read_csv("gold96_submission.csv")
_sample_candidates = list(_BlendPath("/kaggle/input").rglob("sample_submission.csv"))
if not _sample_candidates:
    raise FileNotFoundError("sample_submission.csv not found under /kaggle/input")
_sample_path = [p for p in _sample_candidates if "rogii-wellbore-geology-prediction" in str(p)] or _sample_candidates
sample_sub = _blend_pd.read_csv(_sample_path[0])

def _align_sub(frame, label):
    if "id" not in frame.columns:
        raise KeyError(f"{label} missing id column")
    pred_col = "tvt" if "tvt" in frame.columns else frame.columns[-1]
    compact = frame[["id", pred_col]].rename(columns={pred_col: label})
    if compact["id"].duplicated().any():
        raise ValueError(f"{label} has duplicated ids")
    aligned = sample_sub[["id"]].merge(compact, on="id", how="left", validate="one_to_one")
    if aligned[label].isna().any():
        missing = aligned.loc[aligned[label].isna(), "id"].head(10).tolist()
        raise ValueError(f"{label} missing sample ids: {missing}")
    return _blend_pd.to_numeric(aligned[label], errors="raise").astype(float)

aiden_sel15 = _align_sub(aiden_sel15_sub, "aiden_sel15")
gold96 = _align_sub(gold96_sub, "gold96")
blended = AIDEN_SEL15_WEIGHT * aiden_sel15.to_numpy(float) + GOLD96_WEIGHT * gold96.to_numpy(float)
final_sub = sample_sub.copy()
target_col = "tvt" if "tvt" in final_sub.columns else final_sub.columns[-1]
final_sub[target_col] = blended.astype(float)
final_sub[["id", target_col]].to_csv("submission.csv", index=False)
manifest = {
    "schema_version": "aiwody_physical_gold96_dynamic_blend_v1_w060_clean_nannicha",
    "aiden_sel15_weight": AIDEN_SEL15_WEIGHT,
    "gold96_weight": GOLD96_WEIGHT,
    "rows": int(len(final_sub)),
    "target_column": str(target_col),
    "mean_abs_aiden_sel15_gold96_diff": float(_blend_np.mean(_blend_np.abs(aiden_sel15.to_numpy(float) - gold96.to_numpy(float)))),
    "max_abs_aiden_sel15_gold96_diff": float(_blend_np.max(_blend_np.abs(aiden_sel15.to_numpy(float) - gold96.to_numpy(float)))),
}
_BlendPath("blend_manifest.json").write_text(_blend_json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print(_blend_json.dumps(manifest, indent=2, sort_keys=True))
print("wrote Aiden SEL15 + gold96 blended submission.csv")
# --- Diagnostics and visual summary. This does not change submission.csv. ---
import json as _viz_json
from pathlib import Path as _VizPath

import numpy as _viz_np
import pandas as _viz_pd

try:
    import matplotlib.pyplot as plt
except Exception as _plot_error:
    plt = None
    print(f"matplotlib unavailable: {_plot_error}")

try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    display = None

_diag_frame = sample_sub[["id"]].copy()
_diag_frame["well"] = _diag_frame["id"].astype(str).str[:8]
_diag_frame["row_idx"] = _diag_frame["id"].astype(str).str[9:].astype(int)
_diag_frame["aiden_sel15"] = aiden_sel15.to_numpy(float)
_diag_frame["gold96"] = gold96.to_numpy(float)
_diag_frame["blend"] = final_sub[target_col].to_numpy(float)
_diag_frame["abs_aiden_gold96_diff"] = _viz_np.abs(
    _diag_frame["aiden_sel15"].to_numpy(float) - _diag_frame["gold96"].to_numpy(float)
)

_well_summary = (
    _diag_frame.groupby("well", as_index=False)
    .agg(
        rows=("id", "size"),
        mean_abs_aiden_gold96_diff=("abs_aiden_gold96_diff", "mean"),
        max_abs_aiden_gold96_diff=("abs_aiden_gold96_diff", "max"),
        blend_min=("blend", "min"),
        blend_max=("blend", "max"),
    )
    .sort_values("well")
)

_gold96_manifest_path = _VizPath("gold96_artifact_manifest.json")
_gold96_manifest = None
if _gold96_manifest_path.exists():
    _gold96_manifest = _viz_json.loads(_gold96_manifest_path.read_text(encoding="utf-8"))

_diag = {
    "run_version": "ROGII_aiden_gold96_blend_8107_repeat_2026_06_05",
    "reference": "references/rogii_tree_models_and_physics_blend-8.107",
    "expected_public_score_from_reference_name": 8.107,
    "submission_path": "submission.csv",
    "diagnostics_path": "run_diagnostics_aiden_gold96_blend_8107.json",
    "aiden_sel15_weight": float(AIDEN_SEL15_WEIGHT),
    "gold96_weight": float(GOLD96_WEIGHT),
    "rows": int(len(final_sub)),
    "target_column": str(target_col),
    "mean_abs_aiden_gold96_diff": float(_diag_frame["abs_aiden_gold96_diff"].mean()),
    "max_abs_aiden_gold96_diff": float(_diag_frame["abs_aiden_gold96_diff"].max()),
    "well_summary": _well_summary.to_dict(orient="records"),
    "gold96_artifact_manifest": _gold96_manifest,
    "blend_manifest": manifest,
}
_VizPath("run_diagnostics_aiden_gold96_blend_8107.json").write_text(
    _viz_json.dumps(_diag, indent=2, sort_keys=True), encoding="utf-8"
)
print("Diagnostics: run_diagnostics_aiden_gold96_blend_8107.json")

if display is not None and HTML is not None:
    _cards = [
        ("Rows", f"{len(final_sub):,}", "#264653"),
        ("Aiden weight", f"{AIDEN_SEL15_WEIGHT:.2f}", "#2a9d8f"),
        ("Gold96 weight", f"{GOLD96_WEIGHT:.2f}", "#e9c46a"),
        ("Mean abs diff", f"{_diag['mean_abs_aiden_gold96_diff']:.3f}", "#e76f51"),
        ("Max abs diff", f"{_diag['max_abs_aiden_gold96_diff']:.3f}", "#457b9d"),
    ]
    _card_html = "".join(
        f"""
        <div style="background:{color};color:white;border-radius:8px;padding:14px 16px;min-width:150px;box-shadow:0 1px 3px rgba(0,0,0,.18);">
          <div style="font-size:12px;opacity:.85;text-transform:uppercase;letter-spacing:.04em;">{label}</div>
          <div style="font-size:24px;font-weight:700;margin-top:4px;">{value}</div>
        </div>
        """
        for label, value, color in _cards
    )
    display(HTML(f"<div style='display:flex;gap:12px;flex-wrap:wrap;margin:10px 0 16px 0;'>{_card_html}</div>"))
    display(_well_summary)
else:
    print(_well_summary)

if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(_diag_frame["abs_aiden_gold96_diff"], bins=50, color="#2a9d8f", edgecolor="white")
    axes[0].set_title("Aiden vs Gold96 absolute difference")
    axes[0].set_xlabel("abs diff")
    axes[0].set_ylabel("rows")

    axes[1].bar(
        _well_summary["well"],
        _well_summary["mean_abs_aiden_gold96_diff"],
        color=["#264653", "#e9c46a", "#e76f51", "#457b9d"][: len(_well_summary)],
    )
    axes[1].set_title("Mean component disagreement by well")
    axes[1].set_xlabel("well")
    axes[1].set_ylabel("mean abs diff")
    axes[1].tick_params(axis="x", rotation=30)
    fig.tight_layout()
    plt.show()

    wells = list(_well_summary["well"])
    fig, axes = plt.subplots(len(wells), 1, figsize=(13, max(3, 2.8 * len(wells))), sharex=False)
    axes = _viz_np.atleast_1d(axes)
    for ax, well in zip(axes, wells):
        g = _diag_frame[_diag_frame["well"] == well].sort_values("row_idx")
        stride = max(1, len(g) // 800)
        gg = g.iloc[::stride]
        ax.plot(gg["row_idx"], gg["aiden_sel15"], label="Aiden SEL15", lw=1.0, color="#264653")
        ax.plot(gg["row_idx"], gg["gold96"], label="Gold96", lw=1.0, color="#e9c46a", alpha=0.85)
        ax.plot(gg["row_idx"], gg["blend"], label="Final blend", lw=1.5, color="#e76f51")
        ax.set_title(f"{well}: final blend vs components")
        ax.set_xlabel("row_idx")
        ax.set_ylabel("TVT")
        ax.grid(alpha=0.22)
    axes[0].legend(loc="best")
    fig.tight_layout()
    plt.show()
